<a href="https://colab.research.google.com/github/rakshachahar/flyrank-ml-internship/blob/main/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rakshachahar/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
%pip -q install duckdb pandas scikit-learn

import duckdb
import pandas as pd
import os

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except:
    HF_TOKEN = os.environ.get("HF_TOKEN")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
fact_daily = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"

print("Connected")

Connected


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## Method Choice and Why

I selected Random Forest Classification because it can learn non-linear relationships between multiple search-performance signals without requiring complex feature engineering. This method is appropriate for the Refresh / Content Opportunity Scoring lane because several observable metrics together determine whether a page should be prioritized for review. The model will be compared against the Week 4 baseline using the same data and evaluation approach.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

print("Selected Model: Random Forest Classifier")

Selected Model: Random Forest Classifier


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

## Split Design

A train-test split is used so the model is evaluated on data that was not used during training. The same split is used for both the baseline and the machine learning model to provide a fair comparison.

In [ ]:
sample = con.sql(f"""
SELECT
client_hash_id,
content_hash_id,
gsc_impressions,
gsc_clicks,
gsc_avg_position
FROM {fact_daily}
LIMIT 5000
""").df()

sample["ctr"] = (
    sample["gsc_clicks"] /
    sample["gsc_impressions"].replace(0,1)
)

sample["label"] = (
    sample["gsc_impressions"] >
    sample["gsc_impressions"].median()
).astype(int)

from sklearn.model_selection import train_test_split

X = sample[["gsc_impressions","gsc_clicks","gsc_avg_position","ctr"]]
y = sample["label"]

X_train,X_test,y_train,y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(3750, 4)
(1250, 4)


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

sample = con.sql(f"""
SELECT
client_hash_id,
content_hash_id,
gsc_impressions,
gsc_clicks,
gsc_avg_position
FROM {fact_daily}
LIMIT 5000
""").df()

sample["ctr"] = (
    sample["gsc_clicks"] /
    sample["gsc_impressions"].replace(0,1)
)

sample["label"] = (
    sample["gsc_impressions"] >
    sample["gsc_impressions"].median()
).astype(int)

from sklearn.model_selection import train_test_split

X = sample[["gsc_impressions","gsc_clicks","gsc_avg_position","ctr"]]
y = sample["label"]

X_train,X_test,y_train,y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score
import pandas as pd

# Baseline Model
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

baseline_pred = baseline.predict(X_test)

baseline_accuracy = accuracy_score(
    y_test,
    baseline_pred
)

# Random Forest
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

rf_accuracy = accuracy_score(
    y_test,
    rf_pred
)

comparison = pd.DataFrame({
    "Model":[
        "Baseline",
        "Random Forest"
    ],
    "Accuracy":[
        baseline_accuracy,
        rf_accuracy
    ]
})

display(comparison)

,Model,Accuracy
0,Baseline,0.532
1,Random Forest,1.000


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## Errors and Interpretation

The Random Forest model generally performs better than the baseline because it considers multiple search-performance signals together. Misclassifications may occur when pages have unusual traffic patterns or when important factors are not available in the dataset. The model provides decision-support rather than guaranteed predictions and should be interpreted together with human review.

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    y_test,
    rf_pred
))

importance = pd.DataFrame({
    "Feature":X.columns,
    "Importance":rf.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

display(importance)

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       665
           1       1.00      1.00      1.00       585

    accuracy                           1.00      1250
   macro avg       1.00      1.00      1.00      1250
weighted avg       1.00      1.00      1.00      1250



,Feature,Importance
0,gsc_impressions,0.965338
3,ctr,0.014637
2,gsc_avg_position,0.013397
1,gsc_clicks,0.006628


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.